# 3 - AI insights

This notebook writes the insights in plain English from the numbers, so they update if the data changes.
Then I gave the same numbers to an AI tool to help write the business recommendations (`insights/AI_Insights.md`).

In [1]:
import pandas as pd

df = pd.read_csv("../data/cleaned/sales_model.csv", keep_default_na=False, na_values=[""],
                 parse_dates=["Invoice Date Key"])
res = pd.read_csv("../models/model_results.csv")
imp = pd.read_csv("../models/feature_importance.csv")

In [2]:
lines = []
rev = df["Total Excluding Tax"].sum()
prof = df["Profit"].sum()
lines.append(f"Total revenue is {rev/1e6:.2f}M with {prof/1e6:.2f}M profit ({prof/rev*100:.1f}% margin).")

year = df.groupby("Year")["Total Excluding Tax"].sum()
growth = (year[2015] / year[2013]) ** (1 / 2) - 1
lines.append(f"Revenue grew about {growth*100:.0f}% a year from 2013 to 2015.")

full = df[df["Year"] < 2016]
month = full.groupby(full["Invoice Date Key"].dt.month_name())["Total Excluding Tax"].sum()
lines.append(f"In 2013 - 2015 the best month is {month.idxmax()} and the weakest is {month.idxmin()}.")

top = df.groupby("Description")["Total Excluding Tax"].sum().idxmax()
lines.append(f"The top product is '{top}'.")

state = df.groupby("State Province")["Total Excluding Tax"].sum()
lines.append(f"{state.idxmax()} is the biggest state ({state.max()/rev*100:.0f}% of revenue).")

loss = df[df["Profit"] < 0]
lines.append(f"{len(loss)} lines lost money, total loss {abs(loss['Profit'].sum()):,.0f}.")

if "Sunday" not in df["Weekday"].unique():
    lines.append("There are no sales on Sunday.")

best = res.dropna(subset=["Accuracy"]).sort_values("Accuracy").iloc[-1]
lines.append(f"The best model is {best['Model']} with {best['Accuracy']*100:.1f}% accuracy.")
lines.append(f"The most important input is {imp.iloc[0, 0]}, then {imp.iloc[1, 0]}.")

for l in lines:
    print("-", l)

- Total revenue is 19.88M with 9.92M profit (49.9% margin).
- Revenue grew about 10% a year from 2013 to 2015.
- In 2013 - 2015 the best month is July and the weakest is August.
- The top product is 'Air cushion machine (Blue)'.
- California is the biggest state (44% of revenue).
- 566 lines lost money, total loss 40,502.
- There are no sales on Sunday.
- The best model is Random Forest with 99.6% accuracy.
- The most important input is Quantity, then Unit Price.


In [3]:
with open("../insights/auto_insights.txt", "w") as f:
    f.write("\n".join(lines))